# Config 3: frontier model with skills (Claude)

Config 3 is the tool-access configuration: instead of more parameters or
training, the model can query SAS's own `dictionary.columns` /
`dictionary.tables` through SASPy and get real types, lengths and labels
instead of guessing. That's a different axis of improvement from config1
(small local model, zero-shot) and config2 (same base model, QLoRA
fine-tuned) -- report that asymmetry explicitly rather than present
config3 as "just a smarter model." See the repo root's `PLAN.md` for the
three-way framing and `.claude/skills/sas-data-dictionary/SKILL.md` for
the workflow this notebook runs.

**Runs in Colab or locally, on your own Anthropic API key.** Config3
needs no local compute -- the model is remote -- so the free runtime buys
it nothing, but nothing here is Colab-hostile either and section 0 sets
the environment up either way. What changes between the two:

| | Colab | a box you control |
|---|---|---|
| repo | cloned by section 0 into `/content/` | already checked out |
| `anthropic` SDK | `pip install` each session (section 1) | install once |
| API key | Colab **Secrets** panel, or a `getpass` prompt | `ANTHROPIC_API_KEY` in your shell |
| SAS ground truth (optional) | `apt-get default-jdk` + ODA creds from Secrets | SASPy + Java already set up |
| outputs | ephemeral -- **zip and download them** (last section) | persistent in the repo |

**Two levels of run, and you can stop after the first:**

| | what it needs | what it gives you |
|---|---|---|
| `--no-sas-tool` | an API key, nothing else | the plan's fairer isolate: same model and prompt as configs 1/2, no ground truth |
| full config3 | an API key **and** a SAS/ODA account | the tool-access row -- real types, lengths and labels from SAS |

Without ODA the notebook runs the first automatically, and says so rather
than pretending the tool ran. Both are legitimate rows; only the second
is the one `PLAN.md` calls config 3.

Two ways to do the authoring step, both real config3:

| | `claude_driver.py` (this notebook) | interactive Claude Code |
|---|---|---|
| who authors | `claude-opus-5` over the Anthropic API | the Claude Code session you're typing in |
| needs an API key | yes | no |
| all 20 programs | one command | one program per prompt |
| `elapsed_sec` | measured per program | timed by hand |
| `cost_usd` | computed from the API's `usage` | not metered -- honestly `not recorded`, which is **not** `$0` |

The four tools Claude gets, all backed by scripts this folder already had.
Tool access is real, not pre-baked: Claude decides when to call them.
Handing it a pre-harvested metadata blob would make config3 "a bigger
model with a better prompt", which is the one claim `PLAN.md` says not to
make.

| tool | script behind it | returns |
|---|---|---|
| `sas_column_metadata` | `sas_metadata.py` | SAS's own `dictionary.columns`/`dictionary.tables` -- ground truth |
| `header_comments` | `header_extract.py` | header block + inline `/* ... */` glosses |
| `static_identifier_scan` | `extract.py` | the identifier allow-list, and the only source of macro param names |
| `grep_source` | in the driver | regex over the source, to check a name before writing it |

The schema contract in the system prompt is `schema.PROMPT_SCHEMA_BLOCK`
-- byte for byte what configs 1 and 2 are prompted with -- plus SKILL.md's
step-4 authoring rules, so the three-way comparison isn't confounded by
three different prompts.

Run top to bottom: get the repo -> install the SDK -> load your API key ->
*(optional)* connect SAS -> see the tools individually -> document one
program and read its measured cost -> document all 20 -> the
no-ground-truth run -> score -> download the outputs.

> **This spends your own money.** Every driver cell is a billed API call
> on whatever key you loaded in section 2. The one-program cell runs
> first on purpose: it prints a measured `cost_usd` you can multiply by
> 20 before launching the full run. Nothing is billed until then --
> section 2's key check uses `models.retrieve`, which generates no
> tokens.

## Setup 0: Get the repo and find the environment

Pull the **whole** repo, not just this folder: the cells below need
`../eval-programs/` (the 20 held-out programs + gold) and `../results/`
(`run_eval.py`, `score.py`) as siblings of this folder. Pulling only
`config3-frontier-skills/` leaves those missing and every `../results/...`
call fails with "No such file or directory".

Off Colab this cell changes nothing -- it reports where it is running and
checks the same paths, so the notebook is one file in both places.

`PY` is the interpreter every later cell shells out to. On this project's
own box that is the venv that already has `saspy`; anywhere else it is
just this kernel.

In [ ]:
import json
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/patrickjlong1/sas-llm-paper.git"
CLONE_DIR = "/content/sas-llm-paper"

if IN_COLAB:
    if not os.path.isdir(CLONE_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, CLONE_DIR],
                       check=True)
    os.chdir(os.path.join(CLONE_DIR, "config3-frontier-skills"))


def _imports(python, module):
    """True if `python` can import `module` -- used to pick an interpreter
    rather than assume one."""
    try:
        return subprocess.run([python, "-c", "import " + module],
                              capture_output=True).returncode == 0
    except OSError:
        return False


def sh(*args, check=False):
    """Run a command and stream its output into this cell.

    Used instead of a `!` magic wherever the call sits inside an if/else --
    `!` with a line continuation inside an indented block is the kind of
    thing that works in one Jupyter and not the next, and these cells take
    minutes, so the output has to appear as it goes rather than at the end.
    """
    cmd = [str(a) for a in args if str(a) != ""]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc and check:
        raise SystemExit("command failed (exit %d): %s" % (rc, " ".join(cmd)))
    return rc


# The SAS tool shells out to sas_metadata.py, which needs saspy. Prefer an
# interpreter that already has it; otherwise this kernel, and section 3
# installs saspy into it only if you ask for the ground-truth path.
VENV_PY = "/internal/venvs/main/bin/python3"
PY = VENV_PY if _imports(VENV_PY, "saspy") else sys.executable
os.environ["PY"] = PY

print("running on Colab:", IN_COLAB)
print("cwd:             ", os.getcwd())
print("interpreter:     ", PY)
for need in ("../eval-programs/programs", "../eval-programs/gold", "../results",
             "../.claude/skills/sas-data-dictionary"):
    print("  %-40s %s" % (need, "OK" if os.path.isdir(need) else "MISSING"))

## Setup 1: Install the Anthropic SDK

`claude_driver.py` is the only thing that needs it. `requirements.txt`
also lists `saspy`/`pandas`, which only the optional SAS ground-truth path
in section 3 uses -- they install fine either way and cost nothing if
unused.

In [ ]:
!$PY -m pip install -q -r requirements.txt
!$PY -c "import anthropic; print('anthropic', anthropic.__version__)"


## Setup 2: Your Anthropic API key

**Bring your own key** -- get one at
[console.anthropic.com](https://console.anthropic.com/settings/keys). The
cell below looks in three places, in order, and stops at the first hit:

1. `ANTHROPIC_API_KEY` already in the environment (how you'd do it
   locally: `export ANTHROPIC_API_KEY=sk-ant-...` before starting the
   kernel, so it never touches a notebook cell);
2. **Colab Secrets** -- the key icon in the left sidebar. Add a secret
   named `ANTHROPIC_API_KEY`, paste the value, and toggle *Notebook
   access* on for this notebook. This is the reusable option: the secret
   lives in your Google account, not in the notebook, so the same file
   works for anyone with their own key;
3. a `getpass` prompt, which goes into this kernel's environment only --
   never to disk and never into the saved notebook.

The SDK also reads an `ant auth login` profile from `~/.config/anthropic/`
if you have one; that counts as case 1 and needs nothing here.

**Never paste the key into a cell.** A saved notebook with a key in it is
a leaked key -- this one gets committed to the repo.

The check is a `models.retrieve` call: it proves the key works and prints
the model's real limits without generating a single token, so it costs
nothing.

In [ ]:
import getpass


def load_api_key():
    """Environment -> Colab Secrets -> prompt. Returns where it came from."""
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "environment"
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("ANTHROPIC_API_KEY") or "").strip()
            if key:
                os.environ["ANTHROPIC_API_KEY"] = key
                return "Colab Secrets"
        except Exception as exc:
            # Secret absent, or notebook access not granted for it.
            print("Colab Secrets: %s: %s" % (type(exc).__name__, exc))
    key = getpass.getpass("Anthropic API key (sk-ant-..., Enter to skip): ").strip()
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
        return "getpass prompt"
    return None


source = load_api_key()
print("API key from:", source or "NOWHERE -- the driver cells below will fail")

MODEL = "claude-opus-5"
EFFORT = "high"          # output_config.effort: low | medium | high | xhigh | max
os.environ.update(MODEL=MODEL, EFFORT=EFFORT)
print("model:      ", MODEL, "| effort:", EFFORT)

In [ ]:
# models.retrieve proves the key works and prints the model's real limits
# without generating a token, so this check is free.
import anthropic

model_info = anthropic.Anthropic().models.retrieve(MODEL)
print("model:  ", model_info.id, "|", model_info.display_name)
print("context:", getattr(model_info, "max_input_tokens", "?"), "in /",
      getattr(model_info, "max_tokens", "?"), "out")

## Setup 3 *(optional)*: SAS OnDemand for Academics (ODA)

**Skip this whole section if you don't have a SAS account.** Everything
below still runs; the notebook takes the `--no-sas-tool` path and says so
in every output it writes. You'll get the plan's fairer-isolate row (same
model and prompt as configs 1/2, no ground truth) rather than the
tool-access row.

This is the step config3 benefits from MOST -- it supplies the ground
truth that gives config3 its accuracy advantage over configs 1/2, and
"tool access" is the entire claim the config exists to test.

ODA is free for academic/non-commercial use; sign up at
[welcome.oda.sas.com](https://welcome.oda.sas.com). You need two things:

- **Java**, for SASPy's IOM connection. On this project's box that's the
  portable JRE at `../jre/`; on Colab the cell below `apt-get`s it.
- **Credentials** in `~/.authinfo`. On Colab, put `ODA_USER` and
  `ODA_PASS` in the Secrets panel (same place as the API key) and the
  cell writes the file for you; the values never appear in a saved cell.
  Locally, do it in a terminal instead:

  ```bash
  echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
  chmod 600 ~/.authinfo
  ```

`config/sascfg_personal.py`'s `iomhost` list is filled in for a
US-region/usw2 account; config1's `SETUP.md` has the Europe and Asia
Pacific host names if yours differs.

The cell sets `HAVE_SAS`, and every later cell keys off it -- there is no
way to half-run this and end up with a row that claims tool access it
didn't have.

In [ ]:
import getpass
import shutil

WANT_SAS = True          # set False to force the no-ground-truth path

authinfo = os.path.expanduser("~/.authinfo")


def _java():
    return os.path.exists(os.path.abspath("../jre/bin/java")) or bool(shutil.which("java"))


def _oda_creds():
    """Colab Secrets -> prompt. Returns (user, password) or (None, None)."""
    user = password = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            user = (userdata.get("ODA_USER") or "").strip()
            password = (userdata.get("ODA_PASS") or "").strip()
        except Exception as exc:
            print("Colab Secrets: %s: %s" % (type(exc).__name__, exc))
    if not user:
        user = input("ODA email (Enter to skip SAS entirely): ").strip()
        if not user:
            return None, None
        password = getpass.getpass("ODA password: ")
    return user, password


HAVE_SAS = False
if not WANT_SAS:
    print("WANT_SAS is False -- taking the no-ground-truth path on purpose.")
elif os.path.exists(authinfo):
    print("~/.authinfo found -- leaving it alone")
    HAVE_SAS = True
else:
    user, password = _oda_creds()
    if user:
        with open(authinfo, "w") as fh:
            fh.write("oda user %s password %s\n" % (user, password))
        os.chmod(authinfo, 0o600)
        del password
        print("wrote", authinfo, "(chmod 600)")
        HAVE_SAS = True
    else:
        print("no ODA credentials -- continuing without SAS ground truth")

if HAVE_SAS and not _java():
    if IN_COLAB:
        print("installing a JDK for SASPy's IOM connection (a minute or so)...")
        subprocess.run(["apt-get", "-qq", "install", "-y", "default-jdk"], check=True)
    else:
        print("WARNING: no java on PATH and no ../jre/ -- SASPy cannot connect.")
        HAVE_SAS = _java()

if HAVE_SAS and not _imports(PY, "saspy"):
    print("installing saspy into", PY)
    subprocess.run([PY, "-m", "pip", "install", "-q", "saspy", "pandas"], check=True)

# Every driver cell below interpolates this. Empty string = full config3.
SAS_FLAG = "" if HAVE_SAS else "--no-sas-tool"
os.environ["SAS_FLAG"] = SAS_FLAG
print("\nHAVE_SAS:", HAVE_SAS,
      "-> driver flag:", repr(SAS_FLAG) or "(none: full config3 with ground truth)")
if not HAVE_SAS:
    print("Predictions will be written as the no-ground-truth run. That is a real\n"
          "result -- just not the tool-access row PLAN.md calls config 3.")

---
## The tools, one at a time

Demonstrated on `prog900_estab.sas`, the same program config1's notebook
uses, so all three configs produce comparable output for the same program.

**You don't have to run these three cells.** `claude_driver.py` calls the
same code as tools; these are here so a surprising dictionary entry is
debuggable.

In [ ]:
PROGRAM = "prog900_estab"
SAS_FILE = "../eval-programs/programs/prog900_estab.sas"
print(PROGRAM, "->", SAS_FILE)

### `sas_column_metadata` -- ground truth from SAS itself

Submits the program in a live ODA session, then harvests
`dictionary.columns`/`dictionary.tables` for WORK: real name, type,
length, format, informat, label and observation counts. Takes tens of
seconds (a real SAS session start), and is allowed to fail -- a program
whose hardcoded paths don't resolve outside production often still
materializes useful earlier datasets, so `run_errors` alongside usable
`columns` is a partial success worth keeping, not a failure.

Expect zero non-empty labels on this corpus: `eval-programs/` is
deliberately unlabelled, terse legacy-style SAS. Ground truth still pins
down every `type` and `length`, which is most of what configs 1/2 get
wrong by guessing.

In [ ]:
if not HAVE_SAS:
    print("skipped -- no SAS connection (Setup 3). This is the tool config3 has\n"
          "and configs 1/2 do not, so without it config3 runs as 'same model,\n"
          "same prompt, no ground truth'.")
else:
    sh(PY, "sas_metadata.py", SAS_FILE, "--out", "/tmp/column_metadata.json")
    meta = json.load(open("/tmp/column_metadata.json"))
    print("tables: ", sorted(meta["tables"]))
    print("columns:", len(meta["columns"]),
          "| labelled:", sum(1 for c in meta["columns"] if c["label"]))
    print("errors: ", meta["run_errors"] or "none")
    print(json.dumps(meta["columns"][:3], indent=1))

### `header_comments` and `static_identifier_scan`

Human-authored context (`header_found: false` is the normal result on this
corpus -- it has zero header comments on purpose), then the regex
allow-list of every identifier that demonstrably appears in the source.

In [ ]:
sh(PY, "header_extract.py", SAS_FILE)
sh(PY, "extract.py", SAS_FILE)

---
## Document one program

`claude_driver.py` runs the loop: Claude gets the source plus the four
tools, calls what it needs, and returns the dictionary JSON in
`schema.py`'s shape. The driver then hands that JSON to the same two
scripts the interactive path uses --

- `write_dictionary.py` (`--catalog`): cross-checks every identifier
  against ground truth + the static scan, stamps misses as
  `guardrail_flagged`, and upserts into the local JSONL catalog;
- `save_prediction.py` (`--preds-out`): writes the
  `.pred.json`/`.meta.json` pair `results/score.py` reads, with the
  **measured** `elapsed_sec` and the **real** `cost_usd`.

-- so there's one implementation of validation and cataloguing, and no
`--elapsed-sec 90.0` placeholder to regret later.

If it reports guardrail flags, don't just push: check whether Claude
invented a name or `extract.py`'s regex missed a real one (it does miss
some forms), and fix the JSON before continuing.

Written per program: `claude-runs/<program>.dictionary.json` (the
dictionary), `claude-runs/<program>.run.json` (turns, repair turns,
tool-call counts, token usage, cost), `claude-runs/metadata/<program>.json`
(the harvested ground truth, already in the layout `run_eval.py
--extra-source-dir` reads).

In [ ]:
# Which directory a prediction lands in decides how the results table
# labels it. A run without the SAS tool must never land in the
# tool-access directory -- that would be a row claiming ground truth it
# never had, which is the exact mislabelling ../results/README.md's
# provenance machinery exists to prevent.
PREDS = ("../results/preds/config3-frontier-skills" if HAVE_SAS
         else "../results/preds/config3-frontier-skills-nogt")
RUNS = "claude-runs" if HAVE_SAS else "claude-runs-nogt"
print("this run writes to:", PREDS, "\n              runs:", RUNS)

sh(PY, "claude_driver.py", SAS_FILE, SAS_FLAG,
   "--model", MODEL, "--effort", EFFORT,
   "--out", RUNS, "--catalog", "catalog/", "--preds-out", PREDS)

In [ ]:
run = json.load(open("%s/%s.run.json" % (RUNS, PROGRAM)))
print(json.dumps(run, indent=2))
print("\n--- authored dictionary (first 60 lines) ---")
print("\n".join(open("%s/%s.dictionary.json" % (RUNS, PROGRAM)).read().splitlines()[:60]))

In [ ]:
# What the full run will cost YOUR key, measured rather than guessed.
meta = json.load(open("%s/%s.run.json" % (RUNS, PROGRAM)))
one = meta.get("cost_usd")
n_programs = len(os.listdir("../eval-programs/programs"))

if one is None:
    print("cost_usd is null -- the model that served this turn is not in\n"
          "claude_driver.py's PRICES table. Check console.anthropic.com/pricing\n"
          "before quoting a cost column.")
else:
    print("this program:      $%.4f  (%.1fs, %d API turns, %d repair turns)"
          % (one, meta["elapsed_sec"], meta["api_turns"], meta["repair_turns"]))
    print("projected for %d:  $%.2f" % (n_programs, one * n_programs))
    print("plus the same again if you also run the --no-sas-tool pass below.")
    print("\nThis is a projection from ONE program, not a quote: programs differ in\n"
          "length and in how many tool calls Claude decides to make.")

### Doing this step interactively instead

The driver isn't the only way, and on a single program it isn't
necessarily the best one. In a Claude Code session in this repo, just ask:

> document `eval-programs/programs/prog900_estab.sas` using the
> sas-data-dictionary skill

The skill triggers, runs these same scripts, and writes the same JSON --
then persist it exactly as the driver does:

```bash
python3 write_dictionary.py --program-name prog900_estab \
    --source ../eval-programs/programs/prog900_estab.sas \
    --dictionary /tmp/dictionary.json \
    --column-metadata /tmp/column_metadata.json --catalog catalog/

python3 save_prediction.py --program-name prog900_estab \
    --dictionary /tmp/dictionary.json --elapsed-sec <measured> \
    --used-proc-contents --out ../results/preds/config3-frontier-skills
```

What you lose: the measured cost (an interactive session has no metered
per-call cost, so `cost_usd` is honestly `not recorded` -- **not** `$0`)
and unattended runs. What you gain: you can interrogate a surprising entry
as it's written. Same config either way -- say which one produced the
numbers you report.

---
## Document all 20

**Check the projected cost printed above before running this.** It is 20
billed API calls on your key, plus a repair turn here and there.

One command, one program at a time. A per-program failure (API error, a
dropped SAS session, output that still misses the schema after its repair
attempts) is logged and skipped rather than aborting the run -- same
policy as config1's batch mode -- and the last line lists what failed so
you can re-run just those.

With the SAS tool on, each program opens its own ODA session for the
harvest, so budget on the order of a minute per program, most of it SAS
rather than Claude. Without it the run is much faster and much cheaper.

Add `--limit N` for a smoke test, but a partial run is **not** a result:
`score.py` scores against all 20 gold files and counts every missing
prediction as a program that produced no output.

In [ ]:
sh(PY, "claude_driver.py", "--dir", "../eval-programs/programs", SAS_FLAG,
   "--model", MODEL, "--effort", EFFORT,
   "--out", RUNS, "--catalog", "catalog/", "--preds-out", PREDS)

### The second run the plan asks for: no ground truth

Same model, same prompt, `--no-sas-tool` -- the `sas_column_metadata`
tool is simply not offered, so Claude works from the source text and the
static scan alone, which is what configs 1 and 2 have. That pair of rows
(with tool access / without) is the fairer isolate of model quality,
since config3's ground truth is something the other two configs
structurally cannot get.

It also skips every SAS session, so it's much faster than the run above --
but it is a second full set of billed calls.

**If you skipped Setup 3, the run above already WAS this run** and the
cell below says so instead of billing you twice. There is nothing to
compare it against until you have a SAS account.

In [ ]:
PREDS_NOGT = "../results/preds/config3-frontier-skills-nogt"

if not HAVE_SAS:
    print("skipped -- the run above had no SAS tool, so it already IS the\n"
          "no-ground-truth condition and was written to " + PREDS_NOGT)
else:
    sh(PY, "claude_driver.py", "--dir", "../eval-programs/programs", "--no-sas-tool",
       "--model", MODEL, "--effort", EFFORT,
       "--out", "claude-runs-nogt", "--preds-out", PREDS_NOGT)

---
## Score the run(s)

`run_eval.py` stamps a `.provenance.json` sidecar recording exactly which
eval corpus was scored, and `--table` marks a row **STALE** rather than
printing numbers that no longer refer to the corpus on disk. Config3's own
2026-09-17 run is why that check exists: it read `1.00` on every metric,
while the same predictions scored against the current gold read `0.79`
variable F1 and `0.00` macro F1. See
`../results/outputs/stale-2026-09-17/README.md`.

`--extra-source-dir` points at the ground truth the driver saved, per
`PLAN.md` section 3's "for config 3, PROC CONTENTS output also counts as a
valid source" -- a name that appears in SAS's metadata but not in the
source text is then not a hallucination. The provenance sidecar records
that it was used, so a row scored WITH ground truth is never silently
compared against one scored without it.

The description judge (`../results/llm_judge.py`) is a local Ollama model
(default `gemma3:1b`) -- a different model family from config3, as the
plan requires to avoid self-grading bias. **On Colab there is no Ollama
server**, so leave `--run-judge` off there and run the judge later on a
box that has one; the table prints `not run` for Description score rather
than inventing a number. On a box with Ollama up, add `--run-judge`.

In [ ]:
if HAVE_SAS:
    sh(PY, "../results/run_eval.py", "--config", "config3-frontier-skills",
       "--pred-dir", PREDS, "--extra-source-dir", "claude-runs/metadata",
       "--out-prefix", "../results/outputs/config3-frontier-skills")
else:
    print("skipped -- no tool-access run exists. The next cell scores the\n"
          "no-ground-truth run instead.")

In [ ]:
# The no-ground-truth run: no --extra-source-dir, because there wasn't any.
if os.path.isdir(PREDS_NOGT):
    sh(PY, "../results/run_eval.py", "--config", "config3-frontier-skills-nogt",
       "--pred-dir", PREDS_NOGT,
       "--out-prefix", "../results/outputs/config3-frontier-skills-nogt")
else:
    print("skipped -- no", PREDS_NOGT, "yet")

In [ ]:
# Both rows in one --table call: a --table invocation prints its own header,
# so passing them separately would print two disconnected tables.
#
# Patterns in a rows file are glob()ed against the CURRENT directory, and a
# pattern that matches nothing is skipped silently -- so these are written
# ../results/-relative because this notebook runs from config3's folder, not
# from results/. Getting that wrong prints an empty row, not an error.
label_gt = "Config 3: Claude + SAS tool access (%s, effort=%s)" % (MODEL, EFFORT)
label_no = "Config 3: Claude, no ground truth (%s, effort=%s)" % (MODEL, EFFORT)

rows = []
for label, stem, preds in ((label_gt, "config3-frontier-skills",
                            "config3-frontier-skills"),
                           (label_no, "config3-frontier-skills-nogt",
                            "config3-frontier-skills-nogt")):
    if os.path.exists("../results/outputs/%s.scores.jsonl" % stem):
        rows.append({"label": label,
                     "scores_patterns": ["../results/outputs/%s.scores.jsonl" % stem],
                     "judge_patterns": ["../results/outputs/%s.judge.jsonl" % stem],
                     "meta_dir": "../results/preds/%s" % preds})

if not rows:
    print("nothing scored yet -- run the driver and the scoring cells above.")
else:
    with open("config3_rows.json", "w") as fh:
        json.dump(rows, fh, indent=2)
    print("scoring %d row(s)\n" % len(rows))
    sh(PY, "../results/run_eval.py", "--table", "--rows", "config3_rows.json")

---
## Download the outputs (Colab)

On Colab `/content/` is wiped when the runtime recycles, and **these
files are the run** -- recreating them costs the API spend again. Zip and
download them before you close the tab.

Off Colab this cell is a no-op: everything is already in the repo.

In [ ]:
if not IN_COLAB:
    print("not on Colab -- outputs are already on local disk under\n"
          "  ./%s, ./catalog, ../results/preds, ../results/outputs" % RUNS)
else:
    import shutil
    from google.colab import files

    for src, name in ((RUNS, "config3-runs"),
                      ("catalog", "config3-catalog"),
                      ("../results/preds", "config3-preds"),
                      ("../results/outputs", "config3-outputs")):
        if not os.path.isdir(src):
            print("skip %s (not created this session)" % src)
            continue
        shutil.make_archive("/content/" + name, "zip", src)
        files.download("/content/%s.zip" % name)
        print("downloaded %s.zip <- %s" % (name, src))

## Push the catalog to SAS as real datasets

Writes/replaces `PROGRAM_SUMMARY`, `MACRO_PARAMS`, `DATA_DICTIONARY` and
`COLUMN_METADATA` in the target SAS library (default `SASUSER` on ODA --
pass `--libname`/`--libpath` for a different, permanent, custom-path
library). Each run mirrors the CURRENT full contents of the local catalog,
so it's safe to re-run after documenting more programs; it doesn't append
duplicates.

Check what's in `catalog/` first. The catalog built before the 2026-09-21
rewrite of `../eval-programs/` was moved aside to
`catalog-stale-2026-09-17/`; pushing that one would put documentation for
programs that no longer exist into `SASUSER` under current program names.

Optional -- skip it if you only want the local JSON/catalog output.

In [ ]:
if not HAVE_SAS:
    print("skipped -- pushing the catalog back to SAS needs the same ODA\n"
          "connection Setup 3 didn't get. The local catalog/ JSON is still there.")
elif not os.path.isdir("catalog"):
    print("no catalog/ yet -- run the driver with --catalog first")
else:
    print(sorted(os.listdir("catalog")))
    sh(PY, "push_to_oda.py", "--catalog", "catalog/")

---
## What the driver does and doesn't constrain

- **No decoding constraint by default.** Configs 1 and 2 get the schema as
  prompt *text*; so does config3. `--structured-output` will constrain
  decoding to the JSON schema, which makes schema validity trivially 1.00
  -- fine in a production pipeline, but it would make that column measure
  the harness instead of the model, so it's off by default and worth
  declaring if you turn it on.
- **Repair turns are counted, not hidden.** If the returned JSON misses
  `schema.py`'s shape, the driver re-asks with the validation errors (up
  to `--max-repairs`, default 2) and records `repair_turns` in
  `<program>.run.json`. A config that needed repairs is not the same as
  one that didn't.
- **Cost is priced per turn, at the model that actually served it.**
  `--no-fallbacks` disables the server-side refusal fallback if you'd
  rather a policy decline fail loudly than be answered by another model;
  either way `usage.models_served` records who answered, and an
  unrecognized model id makes `cost_usd` null rather than wrong. The
  `PRICES` table in `claude_driver.py` was checked 2026-09-22 -- re-check
  before quoting the cost column.
- **`--effort`** (`output_config.effort`) is the quality/spend dial:
  `high` is the default and the sweet spot here; `max` costs more for
  marginal gain, `low`/`medium` are the cheaper step-downs to measure
  before assuming you need more.
- **The harvest no longer reports its own scratch tables.**
  `sas_metadata.py` builds `work._sdgcols`/`_sdgtabs` to hold the
  dictionary query's output, and `dictionary.tables` saw them while they
  were being created -- so the ground truth used to list `_SDGCOLS` as one
  of the program's datasets. It's now excluded unconditionally, separately
  from `--exclude-pattern` (which still defaults to excluding nothing,
  because legacy SAS really does name real datasets with a leading
  underscore).

## The asymmetry to report honestly

Config 3's `dictionary.columns` access is real ground truth the other two
configs structurally cannot get: config1 has no SAS connection at all by
design, and config2's fine-tuning happens offline with no live SAS session
at inference time either. If config3 scores much higher on
`type_length_accuracy` or `hallucination_rate`, that may be measuring "has
ground truth" more than "is a better model" -- which is exactly why the
`--no-sas-tool` run exists. Report the pair, and say plainly that
config3's axis is tool access, not model quality alone.

## Running this again, and what it costs

Config3 is the one config whose marginal cost is money rather than time.
Two things follow:

- **Re-running is not free.** `claude-runs*/` and `../results/preds/` are
  the run; the section above downloads them on Colab for exactly this
  reason. Scoring, re-scoring and table-building are all local and free --
  you never need to re-author to change how you score.
- **`--limit N` is for smoke tests only.** `score.py` scores against all
  20 gold files and counts a missing prediction as a program that produced
  no output, so a 5-program run reads as a catastrophic 15-program
  failure, not as a 5-program result.

Knobs worth knowing, all on `claude_driver.py`:

| flag | default | why you'd change it |
|---|---|---|
| `--model` | `claude-opus-5` | a cheaper model is a different results row -- label it |
| `--effort` | `high` | `low`/`medium` cut token spend; `xhigh`/`max` raise it |
| `--limit N` | all | smoke test before committing to 20 |
| `--no-sas-tool` | off | the fairer isolate; set automatically when Setup 3 is skipped |
| `--max-turns` | 16 | ceiling on the tool loop per program |
| `--no-fallbacks` | fallbacks on | make a policy decline fail loudly instead of being answered by another model |

`config1-gemma-cpu/` and `config2-qlora-gpu/` are the notebooks that want
Colab for its free hardware: a CPU runtime and a free T4 respectively.
Config3 wants Colab only as a place to stand.

## Don't confuse this with config1

`config1-gemma-cpu/document_sas.py` is a separate, independent pipeline
(small Gemma model, no ground-truth SAS metadata at all, its own regex
guardrail) used to establish the air-gapped floor for comparison. Don't
mix the two catalogs by hand-editing; they write to different `catalog/`
directories under their own config folders by design, specifically so
config1 and config3 runs never clobber each other.